## Modell A: Verbindung Datenbank und Log-Transformation 'price'

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --------------------------
# SCHRITT 1: Verbindung zur Datenbank herstellen
# --------------------------

# ERSETZEN SIE DIESE PLATZHALTER mit Ihren tatsächlichen Werten
DB_USER = 'Ihr_PostgreSQL_Benutzernname'
DB_PASSWORD = 'Ihr_Passwort'
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'Ihr_Datenbankname'
TABELLE_NAME = 'listings_gesamt' # Angepasst an den Namen im .py-Skript

# Erstellen der Engine/Verbindungszeichenkette
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

# --------------------------
# SCHRITT 2: Daten mit SQL-Abfrage laden (INKL. ROHPREIS UND GEODATEN!)
# --------------------------

sql_query = f"""
SELECT 
    id, 
    price,                      -- NEU: Rohpreis wird geladen
    accommodates, 
    bedrooms, 
    beds,                       -- Hinzugefügt: Für konsistenten Prädiktor-Set
    room_type, 
    city, 
    latitude,                   -- NEU: Geodaten für Feature Engineering
    longitude,                  -- NEU: Geodaten für Feature Engineering
    review_scores_rating 
FROM 
    {TABELLE_NAME};             -- Die WHERE-Klausel entfällt, da das Cleaning-Skript bereits alles bereinigt hat
"""

# Führt die SQL-Abfrage aus und lädt das Ergebnis direkt in einen Pandas DataFrame
try:
    df_ml = pd.read_sql_query(sql_query, engine)
    print(f"Datenbankverbindung erfolgreich. {len(df_ml)} Einträge geladen.")

    # --------------------------
    # SCHRITT 3: Finale Vorbereitung im Notebook (Log-Transformation und Encoding)
    # --------------------------
    
    # 1. LOG-TRANSFORMATION HIER DURCHFÜHREN! (Zielvariable für Modell A)
    df_ml['ln_price'] = np.log(df_ml['price'])
    print("\n✅ Logarithmische Transformation ('ln_price') erstellt.")

    # 2. Imputation der Prädiktoren (z.B. fehlende Schlafzimmer-Angaben)
    # Diese logische Abfolge muss VOR der Regression erfolgen
    for col in ['bedrooms', 'beds', 'review_scores_rating']:
        if col in df_ml.columns and df_ml[col].isnull().any():
            median_val = df_ml[col].median()
            df_ml[col].fillna(median_val, inplace=True)
    print("✅ Imputation der numerischen Prädiktoren abgeschlossen.")

    # 3. Feature Engineering Geodaten (Platzhalter, muss noch implementiert werden)
    # df_ml['distance_to_center_km'] = berechne_distanz(df_ml, ...)
    
    # 4. One-Hot-Encoding starten (Prädiktoren für die Regression)
    df_ml = pd.get_dummies(df_ml, columns=['room_type', 'city'], drop_first=True)
    
    # Anzeigen der ersten Zeilen und der neuen Spalten
    print("\nDaten zur Modellierung bereit (Log-Modell-Pfad):")
    print(df_ml.head())
    
except Exception as e:
    print(f"FEHLER beim Laden der Daten aus PostgreSQL: {e}")
    print("Bitte prüfen Sie Ihre Verbindungsdaten (Benutzername/Passwort/DB-Name).")